# Домашняя работа 1 — доли исходов HMDA

## Подготовка

In [1]:
from pathlib import Path
import sys
import pandas as pd

print("Python:", sys.executable)
print("pandas:", pd.__version__)
print("Рабочая папка:", Path.cwd())
DATA = Path("../data")
assert DATA.is_dir(), "Рядом с папкой ноутбука должна находиться data/"

Python: C:\Users\Кирюха\Desktop\data-analysis-course\.venv\Scripts\python.exe
pandas: 2.2.2
Рабочая папка: C:\Users\Кирюха\Desktop\data-analysis-course\lessons\01-introduction\work


## Основной набор decisions.csv

In [2]:
home_decisions = pd.read_csv(
    DATA / "decisions.csv", dtype={"lei": "string", "county_code": "string"}
)
home_outcomes = home_decisions["action_taken"]
home_n = len(home_decisions)
assert home_n > 0
assert home_outcomes.isna().sum() == 0
assert home_outcomes.isin([1, 2, 3]).sum() == home_n
print(home_decisions.shape)
home_outcomes.value_counts(dropna=False)

(6830, 7)


action_taken
1    5956
3     691
2     183
Name: count, dtype: int64

## H1. Доля отказов

In [3]:
###1 Вычисление количества и доли отказов (код 3) и неотказов (коды 1 и 2)
denial_count = int((home_outcomes == 3).sum())
denial_share = denial_count / home_n
non_denial_count = int((home_outcomes != 3).sum())
non_denial_share = non_denial_count / home_n

###2 Программные проверки диапазона [0, 1] и баланса
assert 0.0 <= denial_share <= 1.0, "Доля отказов должна быть в диапазоне [0, 1]"
assert 0.0 <= non_denial_share <= 1.0, "Доля неотказов должна быть в диапазоне [0, 1]"
assert denial_count + non_denial_count == home_n, "Сумма отказов и неотказов должна равняться home_n"
assert round(denial_share + non_denial_share, 10) == 1.0, "Сумма долей должна равняться 1.0"

print(f"Отказы (код 3): {denial_count} из {home_n} -> доля = {denial_share:.6f} ({denial_share * 100:.2f}%)")
print(f"Неотказы (коды 1, 2): {non_denial_count} из {home_n} -> доля = {non_denial_share:.6f} ({non_denial_share * 100:.2f}%)")
print(f"Проверка баланса: {denial_count} + {non_denial_count} = {denial_count + non_denial_count} (доли: {denial_share + non_denial_share:.6f})")
denial_share

Отказы (код 3): 691 из 6830 -> доля = 0.101171 (10.12%)
Неотказы (коды 1, 2): 6139 из 6830 -> доля = 0.898829 (89.88%)
Проверка баланса: 691 + 6139 = 6830 (доли: 1.000000)


0.10117130307467057

## H2. Одобрение без принятия предложения

In [4]:
approved_not_accepted_count = (home_outcomes == 2).sum()
approved_not_accepted_share = approved_not_accepted_count / home_n

originated_count = (home_outcomes == 1).sum()
originated_share = originated_count / home_n

approved_count = (home_outcomes != 3).sum()
approved_share = approved_count / home_n

diff_pp = (approved_share - originated_share) * 100

assert 0 <= approved_not_accepted_share <= 1
assert originated_count + approved_not_accepted_count + denial_count == home_n
assert round(approved_share - originated_share, 6) == round(approved_not_accepted_share, 6)

print(f"Одобрено без принятия(2): {approved_not_accepted_count} из {home_n} ({approved_not_accepted_share:.6f} ({approved_not_accepted_share * 100:.2f}%))")
print(f"Одобрены и выданы (1): {originated_count} ({originated_share * 100:.2f}%)")
print(f"Все одобрения (коды 1 и 2): {approved_count} ({approved_share * 100:.2f}%)")
print(f"разность: {diff_pp:.2f}") 
approved_not_accepted_share

Одобрено без принятия(2): 183 из 6830 (0.026794 (2.68%))
Одобрены и выданы (1): 5956 (87.20%)
Все одобрения (коды 1 и 2): 6139 (89.88%)
разность: 2.68


0.026793557833089312

## H3. Объяснение инструментов
- **Series и DataFrame, пример:** `DataFrame` — двумерная таблица целиком (`home_decisions`), а `Series` — один её столбец с индексом (`home_outcomes = home_decisions["action_taken"]`).
- **nrows и head, пример:** Параметр `nrows=5` в `pd.read_csv(DATA / "decisions.csv", nrows=5)` считывает с диска только первые 5 строк файла при загрузке, а метод `home_decisions.head(5)` возвращает первые 5 строк уже полностью загруженной в память таблицы.
- **Метод и атрибут, пример:** Атрибут обращается к хранимому свойству объекта без скобок (`home_decisions.shape` возвращает `(6830, 7)`), а метод вызывается со скобками и выполняет вычисление (`home_outcomes.value_counts(dropna=False)`).
- **Ссылка на документацию выбранного метода и уточнённое свойство:** Документация метода [`pandas.Series.value_counts`](https://pandas.pydata.org/docs/reference/api/pandas.Series.value_counts.html). Уточнённое свойство: по умолчанию у метода установлен параметр `dropna=True`, из-за чего пропущенные значения (`NaN`/`<NA>`) автоматически исключаются из подсчёта; передача `dropna=False` (как в ячейке ноутбука) принудительно включает подсчёт количества пропусков отдельной строкой, а параметр `normalize=True` возвращает относительные частоты (доли) вместо абсолютных количеств.

## Дополнительная часть

## Исходный снимок для H+1

In [5]:
raw = pd.read_csv(
    DATA / "hmda_2023_dc_raw.csv",
    dtype={"lei": "string", "county_code": "string"},
)
raw_outcomes = raw["action_taken"]
raw_n = len(raw)
assert raw_n > 0
print(raw.shape)
raw_outcomes.value_counts(dropna=False)

FileNotFoundError: [Errno 2] No such file or directory: '..\\data\\hmda_2023_dc_raw.csv'

## H+1. Доли выдач в двух наборах

## Текстовые коды для H+2

## H+2. Сравнение текста с числом и текстом